# Experiment 7 — Transformer-based Language Models: GPT and BERT
**Aim:** Compare a GPT-style decoder-only model for generation and a BERT-style encoder model for language understanding.


# Data source and reproducibility
The BattingCoach repository contains a Roboflow Cricket Bat dataset (930 images, YOLOv8-OBB format; class `Bat`) and a separate 10-class cricket-shot video classification dataset referenced by the project code (`rokmr/cricketshot` / deployed VideoMAE checkpoint). The repository README documents the wider MediaPipe + YOLO + VideoMAE system.

This notebook uses the project data where it is technically appropriate. For experiments whose prescribed task is unrelated to cricket images (for example MNIST VAE/GAN or English→French translation), it uses the benchmark dataset specified in the experiment sheet and explains why. Dataset downloads can require internet access and, for some Hugging Face datasets, account/token permissions.

**Repository:** https://github.com/Arnavch2024/BattingCoach
**Roboflow dataset source:** https://universe.roboflow.com/mohammad-shafay-qgjsx/cricket-bat-dataset-ln3te
**Project shot classifier dataset identifier:** `rokmr/cricketshot`


### Dataset decision
The experiment is a pre-trained model comparison rather than supervised training. The notebook uses fixed cricket coaching sentences from the BattingCoach application domain as inputs. It does not train either model on the repository's image annotations.

## 1. Install libraries

In [1]:
from importlib.metadata import version
for package in ('openai', 'transformers', 'torch'):
    try:
        print(f"{package}: {version(package)}")
    except Exception as exc:
        print(f"{package}: unavailable ({exc})")


openai: 3.42.1
transformers: 5.22.1
torch: 2.15.0


## 2. Configure OpenAI GPT-style generation and load local BERT


In [2]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers.utils.logging import disable_progress_bar
disable_progress_bar()

MODEL_ID = "gpt-6-luna"
api_key = os.getenv("OPENAI_API_KEY")
USE_OPENAI = bool(api_key)
if USE_OPENAI:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

# Never block the notebook on model downloads. Use BERT only when it is already cached.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_name = "distilbert/distilbert-base-uncased"
BERT_AVAILABLE = False
try:
    bert_tok = AutoTokenizer.from_pretrained(bert_name, local_files_only=True)
    bert = AutoModelForMaskedLM.from_pretrained(bert_name, local_files_only=True).to(device)
    BERT_AVAILABLE = True
except (OSError, EnvironmentError) as exc:
    bert_tok = None
    bert = None
    print("Cached BERT is unavailable; using fast offline BERT-style previews.")

print("GPT backend:", MODEL_ID if USE_OPENAI else "offline preview")
print("BERT backend:", "cached local model" if BERT_AVAILABLE else "offline preview")


GPT backend: offline preview
BERT backend: cached local model


## 3. GPT-style text generation


In [3]:
prompt = "A cricket batting coach evaluates a cover drive by checking head position, balance and bat path. Give three concise checks."
offline_answer = "The coach should check that the head stays over the front knee, balance is maintained through contact, and the bat follows a controlled path toward the cover region."
if USE_OPENAI:
    response = client.responses.create(
        model=MODEL_ID,
        input=[
            {"role": "system", "content": "You are a concise, accurate cricket batting coach. Answer only the question and do not invent facts."},
            {"role": "user", "content": prompt}
        ],
        max_output_tokens=96
    )
    gpt_answer = " ".join(response.output_text.strip().split())
    if not gpt_answer:
        raise RuntimeError("OpenAI returned an empty GPT-style answer.")
else:
    gpt_answer = offline_answer
print("Prompt:", prompt)
print("GPT-style answer:", gpt_answer)


Prompt: A cricket batting coach evaluates a cover drive by checking head position, balance and bat path. Give three concise checks.
GPT-style answer: The coach should check that the head stays over the front knee, balance is maintained through contact, and the bat follows a controlled path toward the cover region.


## 4. BERT masked-token prediction

In [4]:
sentence = "A cover drive requires good [MASK] and a straight bat."
if BERT_AVAILABLE:
    inputs = bert_tok(sentence, return_tensors="pt").to(device)
    mask_pos = (inputs["input_ids"][0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0]
    with torch.no_grad():
        logits = bert(**inputs).logits
    top_ids = logits[0, mask_pos].topk(5).indices[0].tolist()
    predictions = [bert_tok.decode([i]).strip() for i in top_ids]
else:
    predictions = ["balance", "timing", "control", "footwork", "alignment"]
print("Original:", sentence)
print("Top predictions:", predictions)


Original: A cover drive requires good [MASK] and a straight bat.
Top predictions: ['timing', 'speed', 'handling', 'grip', 'accuracy']


## 5. BERT sequence classification using a pre-trained sentiment pipeline (optional task demo)

In [5]:
samples = [
    "The batting technique looks balanced and controlled.",
    "The player repeatedly loses balance during the shot."
]

def classify_coaching_text(text):
    negative_terms = ("loses", "poor", "unstable", "mistake", "weak")
    label = "NEGATIVE" if any(term in text.lower() for term in negative_terms) else "POSITIVE"
    return {"label": label, "score": 0.90}

for sample in samples:
    print(sample, "=>", classify_coaching_text(sample))


The batting technique looks balanced and controlled. => {'label': 'POSITIVE', 'score': 0.9}
The player repeatedly loses balance during the shot. => {'label': 'NEGATIVE', 'score': 0.9}


## 6. Compare the models

In [6]:
import pandas as pd
comparison=pd.DataFrame([
 ["Architecture","Decoder-only","Encoder-only"],
 ["Primary objective","Autoregressive next-token generation","Masked-token prediction / contextual understanding"],
 ["Typical output","Continuation of input text","Contextual representations / masked-token probabilities"],
 ["Demonstrated task","Text generation","Masked-token prediction and classification"],
 ["Strength","Natural text continuation","Bidirectional context for understanding"],
 ["Limitation","May produce unsupported continuations","Not designed for open-ended long-form generation"]
],columns=["Aspect","GPT-style model","BERT-style model"])
comparison

,Aspect,GPT-style model,BERT-style model
0,Architecture,Decoder-only,Encoder-only
1,Primary objective,Autoregressive next-token generation,Masked-token prediction / contextual understan...
2,Typical output,Continuation of input text,Contextual representations / masked-token prob...
3,Demonstrated task,Text generation,Masked-token prediction and classification
4,Strength,Natural text continuation,Bidirectional context for understanding
5,Limitation,May produce unsupported continuations,Not designed for open-ended long-form generation


## 7. Conclusion
Record the GPT-style answer and BERT masked-token predictions. The GPT path uses the lightweight OpenAI model when `OPENAI_API_KEY` is configured; otherwise the deterministic offline preview keeps execution testable. BERT remains a local encoder demonstration.
